### Service 2: Semantic Query

* One service must allow users to ask questions that are resolved through a semantic search (or a hybrid approach, such as lexical search followed by semantic search).
* You may use the datasets introduced in class, or choose your own dataset. 

If you use your own dataset:
* Please **limit file sizes to 40 MB**, so it can be easily shared via GitHub. Note that GitHub warns about files over 50 MB and we generally want to avoid uploading large files.
* **Do not expect us to run the code use to produce embeddings** in the repository. You can include the code used to produce the embeddings, but we ask you to describe your embedding process in the project’s README file.
* Use a [ChromaDB instance with file persistence](https://docs.trychroma.com/docs/run-chroma/persistent-client). This is similar to the first implementation used in class but smaller and easier to host than the Docker-based version.
* If your app needs to access structured data (e.g., to enrich query results), you may use CSV files read with pandas as a back end.
* Please do not use SQLite. We did not include a SQLite library in your environment.

In [28]:
#####Service2#####
#import csv
import re
import pandas as pd



#load article db
article_db_filename="scraped_articles"

scraped_articles = pd.read_csv(f"{article_db_filename}.csv")

print(scraped_articles)

                                   uuid  \
0  948be2f1-bb49-4a84-b931-5cbf24667b10   
1  3170d89c-0794-42e3-b65e-7bc2a52a5908   
2  7c9cd005-a22d-4460-8f15-a83419bccf54   

                                               title  \
0  JTN Networks Acquires Post Millennial And Huma...   
1            BYU Basketball's Kennard Davis Arrested   
2  Miami Dolphins vs. Washington Commanders in Ma...   

                        source  \
0    deadline.com (2025-11-14)   
1  usmagazine.com (2025-11-14)   
2        CBS News (2025-11-14)   

                                                 url  \
0  https://deadline.com/2025/11/right-wing-media-...   
1  https://www.usmagazine.com/entertainment/news/...   
2  https://www.cbsnews.com/video/miami-dolphins-v...   

                                                text  
0  JTN Networks, the parent company of Just the N...  
1  Brigham Young University men’s basketball star...  
2  Miami Dolphins vs. Washington Commanders in Ma...  


In [29]:
#clean csv file as needed
clean_url=False #or True

if clean_url == True:
    # fix markdown links
    def clean_url(url):
        m = re.search(r"\((https?://[^\)]+)\)", url)
        return m.group(1) if m else url

    scraped_articles["url"] = scraped_articles["url"].apply(clean_url)

    # remove rows where scraping failed
    scraped_articles = scraped_articles[~scraped_articles["text"].str.startswith("newspaper3k failed")]

    scraped_articles.to_csv(f"{article_db_filename}_ready.csv", index=False)
if clean_url == False:
    scraped_articles.to_csv(f"{article_db_filename}_ready.csv", index=False)
else:
    print("[MSG] clean_url must be True or False")    
print(scraped_articles)

                                   uuid  \
0  948be2f1-bb49-4a84-b931-5cbf24667b10   
1  3170d89c-0794-42e3-b65e-7bc2a52a5908   
2  7c9cd005-a22d-4460-8f15-a83419bccf54   

                                               title  \
0  JTN Networks Acquires Post Millennial And Huma...   
1            BYU Basketball's Kennard Davis Arrested   
2  Miami Dolphins vs. Washington Commanders in Ma...   

                        source  \
0    deadline.com (2025-11-14)   
1  usmagazine.com (2025-11-14)   
2        CBS News (2025-11-14)   

                                                 url  \
0  https://deadline.com/2025/11/right-wing-media-...   
1  https://www.usmagazine.com/entertainment/news/...   
2  https://www.cbsnews.com/video/miami-dolphins-v...   

                                                text  
0  JTN Networks, the parent company of Just the N...  
1  Brigham Young University men’s basketball star...  
2  Miami Dolphins vs. Washington Commanders in Ma...  


In [30]:
#configure Chroma to save and load the database
import chromadb
import pandas as pd

# Load cleaned CSV
df = pd.read_csv("scraped_articles_ready.csv")

# Create or load persistent ChromaDB
client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_or_create_collection(
    name="articles",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for embeddings
)

# Add all rows to Chroma
collection.add(
    documents=df["text"].tolist(),
    metadatas=df[["uuid", "title", "url", "source"]].to_dict(orient="records"),
    ids=df["uuid"].tolist()
)


In [32]:
#query for news article by embeddings. e.g. news related to the US
results = collection.query(
    query_texts=["usa"],
    n_results=3
)

print(results)

{'ids': [['7c9cd005-a22d-4460-8f15-a83419bccf54', '948be2f1-bb49-4a84-b931-5cbf24667b10', 'cdeb21ce-345c-4a39-90b1-0c956bbd81ec']], 'embeddings': None, 'documents': [["Miami Dolphins vs. Washington Commanders in Madrid NFL fans are in Madrid, Spain, for a historic match between the Miami Dolphins and the Washington Commanders. CBS News Miami's Mike Cugno reports.", 'JTN Networks, the parent company of Just the News, has acquired the Human Events and The Post Millennial, uniting news brands on the right.\n\nHuman Events, which was founded in 1944 and has more recently featured columns and podcasts from Jack Posobiec, the MAGA influencer, will end daily publication this month and will be transformed into a virtual and live events platform, including ticket, music and movie sales and daily event programming, according to JTN Network’s CEO Mark Meckler. John Solomon, the founder of Just the News, will serve as chief strategy and content officer and board chairman.\n\nThe Post Millennial, a